# Eric Rumsfeld's Official ROY Projection Statistical Model

##### Eric Rumsfeld
##### December 3rd, 2025
##### Submission for review to the NBA Future Analytics Stars Program

## Table of Contents
### 1. Data Acquisition
##### a) Making Imports and Loading NBA API Data
##### b) Loading and Managing Current Season Data (2025-26)
###### bI.) Calculating New Feature Columns: PRA, GP_pct, Opp_met
###### bII.) Selecting the Current Rookies
###### bIII.) Handling Undrafted Players
###### bIV.) Saving the Current Season to a CSV
##### c) Loading Historical Data from NBA API (Seasons 2010-11 through 2024-25)
###### cI-cXIX.) follows a similar process to bI-bIV -- reference below
##### d) Specifying Who Received ROY Votes (Seasons 2010-11 through 2024-25)
###### dI.) Inputting Vote Shares
###### dII.) Correcting for Undrafted Players
###### dIII.) Saving Master Historical ROY Dataframe as CSV

### 2. Feature Engineering
##### a) Selecting the Final Features
##### b) Handling Missing Data
##### c) Creating X & y Sets

### 3. Model Training
##### a) Data Split
##### b) Completing Training

### 4. Evaluation
##### a) Calculating Metrics to Understand Performance: MAE, R^2, Spearman Rank
##### b) Extract Feature Importance

### 5. Final Predictions
##### a) Prepping Current Rookie Matrix
##### b) Predicting Probabilities
##### c) Exporting to CSV

## 1. Data Acquisition

### 1a) Starting with all Necessary Imports & Loading NBA API

In [2]:
# Making All Necessary Imports
import pandas as pd
import numpy as np
from nba_api.stats.endpoints import *
import time
from nba_api.stats.endpoints import LeagueDashPlayerStats
from nba_api.stats.endpoints import CommonPlayerInfo

### 1b) Loading Current Season (2025-26) Data from NBA API

In [3]:
# Commented this out so it runs smoothly for you
'''
time1 = time.time()
# Pulling the season stats
stats_df_2025_26 = LeagueDashPlayerStats(                ### edit year
    season="2025-26",                                    ### edit year
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame"
).get_data_frames()[0]

player_ids = stats_df_2025_26["PLAYER_ID"].unique()     ### edit year

experience_data = []

for pid in player_ids:
    try:
        info = CommonPlayerInfo(player_id=pid)
        df = info.get_data_frames()[0]
        exp1 = df["SEASON_EXP"].iloc[0]
        exp2 = df["DRAFT_NUMBER"].iloc[0]
        experience_data.append((pid, exp1, exp2))
        time.sleep(0.3)    # ✅ Rate-limit protection
    except Exception as e:
        print(f"Failed on {pid}: {e}")
        experience_data.append((pid, None))

exp_df_2025_26 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "SEASON_EXP", "DRAFT_NUMBER"])              ### edit year

# Merge with stats
full_df_2025_26 = stats_df_2025_26.merge(exp_df_2025_26, on="PLAYER_ID", how="left")             ### edit year


time2 = time.time()
timediff = time2-time1
tdm = timediff/60
print('This took', round(tdm,3), 'minutes to run.')

# This is all the progress you need to make here.
# Commented this out so it runs smoothly for you
'''

'\ntime1 = time.time()\n# Pulling the season stats\nstats_df_2025_26 = LeagueDashPlayerStats(                ### edit year\n    season="2025-26",                                    ### edit year\n    season_type_all_star="Regular Season",\n    per_mode_detailed="PerGame"\n).get_data_frames()[0]\n\nplayer_ids = stats_df_2025_26["PLAYER_ID"].unique()     ### edit year\n\nexperience_data = []\n\nfor pid in player_ids:\n    try:\n        info = CommonPlayerInfo(player_id=pid)\n        df = info.get_data_frames()[0]\n        exp1 = df["SEASON_EXP"].iloc[0]\n        exp2 = df["DRAFT_NUMBER"].iloc[0]\n        experience_data.append((pid, exp1, exp2))\n        time.sleep(0.3)    # ✅ Rate-limit protection\n    except Exception as e:\n        print(f"Failed on {pid}: {e}")\n        experience_data.append((pid, None))\n\nexp_df_2025_26 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "SEASON_EXP", "DRAFT_NUMBER"])              ### edit year\n\n# Merge with stats\nfull_df_2025_26 = stats_df_2

#### bI.) Creating and calculating the extra column stats I need: PRA, GP_pct, and Opp_met
###### These stats correlate to: Points+Rebounds+Assists, Percentage of Games Played so far, and a self-created Opportunity Metric for the rest of the season

In [4]:
#print(full_df_2025_26.columns.tolist())

In [5]:
# Commented this out so it runs smoothly for you
'''
# PRA calculation
full_df_2025_26["PRA"] = full_df_2025_26["PTS"] + full_df_2025_26["REB"] + full_df_2025_26["AST"]

# Games in the season for teams. Source: https://www.nba.com/stats/teams/traditional
full_df_2025_26["TOT_GAMES"] = 0
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="MIL", "TOT_GAMES"] = 22
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="WAS", "TOT_GAMES"] = 20
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="DEN", "TOT_GAMES"] = 20
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="HOU", "TOT_GAMES"] = 18
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="IND", "TOT_GAMES"] = 21
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="OKC", "TOT_GAMES"] = 21
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="UTA", "TOT_GAMES"] = 20
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="PHI", "TOT_GAMES"] = 20
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="LAL", "TOT_GAMES"] = 20
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="GSW", "TOT_GAMES"] = 21
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="BOS", "TOT_GAMES"] = 21
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="MIA", "TOT_GAMES"] = 21
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="ORL", "TOT_GAMES"] = 21
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="DAL", "TOT_GAMES"] = 22
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="MIN", "TOT_GAMES"] = 21
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="CHA", "TOT_GAMES"] = 21
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="NYK", "TOT_GAMES"] = 20
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="ATL", "TOT_GAMES"] = 22
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="DET", "TOT_GAMES"] = 21
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="CHI", "TOT_GAMES"] = 20
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="BKN", "TOT_GAMES"] = 20
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="SAS", "TOT_GAMES"] = 20
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="POR", "TOT_GAMES"] = 21
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="LAC", "TOT_GAMES"] = 21
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="TOR", "TOT_GAMES"] = 22
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="NOP", "TOT_GAMES"] = 22
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="MEM", "TOT_GAMES"] = 22
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="CLE", "TOT_GAMES"] = 22
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="PHX", "TOT_GAMES"] = 22
full_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="SAC", "TOT_GAMES"] = 21

# % Games played by player
full_df_2025_26["GP_pct"] = full_df_2025_26["GP"] / full_df_2025_26["TOT_GAMES"]
print(full_df_2025_26.head())
# Commented this out so it runs smoothly for you
'''

'\n# PRA calculation\nfull_df_2025_26["PRA"] = full_df_2025_26["PTS"] + full_df_2025_26["REB"] + full_df_2025_26["AST"]\n\n# Games in the season for teams. Source: https://www.nba.com/stats/teams/traditional\nfull_df_2025_26["TOT_GAMES"] = 0\nfull_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="MIL", "TOT_GAMES"] = 22\nfull_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="WAS", "TOT_GAMES"] = 20\nfull_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="DEN", "TOT_GAMES"] = 20\nfull_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="HOU", "TOT_GAMES"] = 18\nfull_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="IND", "TOT_GAMES"] = 21\nfull_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="OKC", "TOT_GAMES"] = 21\nfull_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="UTA", "TOT_GAMES"] = 20\nfull_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="PHI", "TOT_GAMES"] = 20\nfull_df_2025_26.loc[full_df_2025_26["TEAM_ABBREVIATION"]=="LAL", "TOT_GAMES"] = 20\nf

In [6]:
# Commented this out so it runs smoothly for you
'''
full_df_2025_26["Opp_met"] = (full_fr_df_2025_26["MIN"]) * (full_fr_df_2025_26["GP_pct"]) * (1+((full_fr_df_2025_26["PLUS_MINUS"])/10))
# comment for spacing
# Commented this out so it runs smoothly for you
'''

'\nfull_df_2025_26["Opp_met"] = (full_fr_df_2025_26["MIN"]) * (full_fr_df_2025_26["GP_pct"]) * (1+((full_fr_df_2025_26["PLUS_MINUS"])/10))\n# comment for spacing\n# Commented this out so it runs smoothly for you\n'

In [7]:
#print(full_df_2025_26["Opp_met"])

#### b.II) Selecting Current Rookies

In [8]:
# Commented this out so it runs smoothly for you
'''
rookies_2025_26 = full_df_2025_26[full_df_2025_26["SEASON_EXP"] == 0]
# Commented this out so it runs smoothly for you
'''

'\nrookies_2025_26 = full_df_2025_26[full_df_2025_26["SEASON_EXP"] == 0]\n# Commented this out so it runs smoothly for you\n'

#### b.III) Handling undrafted players

In [9]:
# Commented this out so it runs smoothly for you
'''
rookies_2025_26.loc[rookies_2025_26["DRAFT_NUMBER"]=="Undrafted", "DRAFT_NUMBER"] = 61
rookies_2025_26["DRAFT_NUMBER"] = rookies_2025_26["DRAFT_NUMBER"].fillna(61)
rookies_2025_26["DRAFT_NUMBER"] = pd.to_numeric(
    rookies_2025_26["DRAFT_NUMBER"],
    errors="coerce"
)
rookies_2025_26["DRAFT_BUCKET"] = rookies_2025_26["DRAFT_NUMBER"].clip(upper=15)
# Commented this out so it runs smoothly for you
'''

'\nrookies_2025_26.loc[rookies_2025_26["DRAFT_NUMBER"]=="Undrafted", "DRAFT_NUMBER"] = 61\nrookies_2025_26["DRAFT_NUMBER"] = rookies_2025_26["DRAFT_NUMBER"].fillna(61)\nrookies_2025_26["DRAFT_NUMBER"] = pd.to_numeric(\n    rookies_2025_26["DRAFT_NUMBER"],\n    errors="coerce"\n)\nrookies_2025_26["DRAFT_BUCKET"] = rookies_2025_26["DRAFT_NUMBER"].clip(upper=15)\n# Commented this out so it runs smoothly for you\n'

In [10]:
#print(rookies_2025_26)
#print('This should say ~68:', len(rookies_2025_26))

#### b.IV) Saving the dataframe to a csv

In [11]:
# Commented this out so it runs smoothly for you
'''
rookies_2025_26.to_csv("rookies_2025_26.csv", index=False)
# Commented this out so it runs smoothly for you
'''

'\nrookies_2025_26.to_csv("rookies_2025_26.csv", index=False)\n# Commented this out so it runs smoothly for you\n'

## c) Loading Historical Data from NBA API

### cI.) 2010-11 Season
##### Rookie of the Year: Blake Griffin

In [12]:
# Commented this out so it runs smoothly for you
'''
# Pulling the season stats
stats_df_2010_11 = LeagueDashPlayerStats(
    season="2010-11",
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame"
).get_data_frames()[0]

player_ids = stats_df_2010_11["PLAYER_ID"].unique()

experience_data = []

for pid in player_ids:
    try:
        info = CommonPlayerInfo(player_id=pid)
        df = info.get_data_frames()[0]
        exp = df["DRAFT_NUMBER"].iloc[0]
        experience_data.append((pid, exp))
        time.sleep(0.4)    # ✅ Rate-limit protection
    except Exception as e:
        print(f"Failed on {pid}: {e}")
        experience_data.append((pid, None))

exp_df_2010_11 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])

# Merge with stats
full_df_2010_11 = stats_df_2010_11.merge(exp_df_2010_11, on="PLAYER_ID", how="left")
# Commented this out so it runs smoothly for you
'''

'\n# Pulling the season stats\nstats_df_2010_11 = LeagueDashPlayerStats(\n    season="2010-11",\n    season_type_all_star="Regular Season",\n    per_mode_detailed="PerGame"\n).get_data_frames()[0]\n\nplayer_ids = stats_df_2010_11["PLAYER_ID"].unique()\n\nexperience_data = []\n\nfor pid in player_ids:\n    try:\n        info = CommonPlayerInfo(player_id=pid)\n        df = info.get_data_frames()[0]\n        exp = df["DRAFT_NUMBER"].iloc[0]\n        experience_data.append((pid, exp))\n        time.sleep(0.4)    # ✅ Rate-limit protection\n    except Exception as e:\n        print(f"Failed on {pid}: {e}")\n        experience_data.append((pid, None))\n\nexp_df_2010_11 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])\n\n# Merge with stats\nfull_df_2010_11 = stats_df_2010_11.merge(exp_df_2010_11, on="PLAYER_ID", how="left")\n# Commented this out so it runs smoothly for you\n'

### cII.) 2011-12 Season
##### Rookie of the Year: Kyrie Irving

In [13]:
# Commented this out so it runs smoothly for you
'''
time1 = time.time()
# Pulling the season stats
stats_df_2011_12 = LeagueDashPlayerStats(
    season="2011-12",
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame"
).get_data_frames()[0]

player_ids = stats_df_2011_12["PLAYER_ID"].unique()

experience_data = []

for pid in player_ids:
    try:
        info = CommonPlayerInfo(player_id=pid)
        df = info.get_data_frames()[0]
        exp = df["DRAFT_NUMBER"].iloc[0]
        experience_data.append((pid, exp))
        time.sleep(0.4)    # ✅ Rate-limit protection
    except Exception as e:
        print(f"Failed on {pid}: {e}")
        experience_data.append((pid, None))

exp_df_2011_12 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])

# Merge with stats
full_df_2011_12 = stats_df_2011_12.merge(exp_df_2011_12, on="PLAYER_ID", how="left")


time2 = time.time()
timediff = time2-time1
tdm = timediff/60
print('This took', timediff, 'seconds to run. Which is equal to', tdm, 'minutes.')

# This is all the progress you need to make here.
# Commented this out so it runs smoothly for you
'''

'\ntime1 = time.time()\n# Pulling the season stats\nstats_df_2011_12 = LeagueDashPlayerStats(\n    season="2011-12",\n    season_type_all_star="Regular Season",\n    per_mode_detailed="PerGame"\n).get_data_frames()[0]\n\nplayer_ids = stats_df_2011_12["PLAYER_ID"].unique()\n\nexperience_data = []\n\nfor pid in player_ids:\n    try:\n        info = CommonPlayerInfo(player_id=pid)\n        df = info.get_data_frames()[0]\n        exp = df["DRAFT_NUMBER"].iloc[0]\n        experience_data.append((pid, exp))\n        time.sleep(0.4)    # ✅ Rate-limit protection\n    except Exception as e:\n        print(f"Failed on {pid}: {e}")\n        experience_data.append((pid, None))\n\nexp_df_2011_12 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])\n\n# Merge with stats\nfull_df_2011_12 = stats_df_2011_12.merge(exp_df_2011_12, on="PLAYER_ID", how="left")\n\n\ntime2 = time.time()\ntimediff = time2-time1\ntdm = timediff/60\nprint(\'This took\', timediff, \'seconds to run. Which is eq

### cIII.) 2012-13 Season
##### Damian Lillard

In [14]:
# Commented this out so it runs smoothly for you
'''
time1 = time.time()
# Pulling the season stats
print('Pulling the season stats...')
stats_df_2012_13 = LeagueDashPlayerStats(                ### edit year
    season="2012-13",                                    ### edit year
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame"
).get_data_frames()[0]


print('Recording Common Player Info...')
player_ids = stats_df_2012_13["PLAYER_ID"].unique()     ### edit year

experience_data = []

for pid in player_ids:
    try:
        info = CommonPlayerInfo(player_id=pid)
        df = info.get_data_frames()[0]
        exp = df["DRAFT_NUMBER"].iloc[0]
        experience_data.append((pid, exp))
        time.sleep(0.4)    # ✅ Rate-limit protection
    except Exception as e:
        print(f"Failed on {pid}: {e}")
        experience_data.append((pid, None))


print('Merging dataframes...')

# Creating the experience dataframe
exp_df_2012_13 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year
# Merge with stats
full_df_2012_13 = stats_df_2012_13.merge(exp_df_2012_13, on="PLAYER_ID", how="left")             ### edit year


time4 = time.time()
timediff = time4-time1
tdm = timediff/60
print('This took', round(tdm,3), 'minutes.')

# This is all the progress you need to make here.
# Commented this out so it runs smoothly for you
'''

'\ntime1 = time.time()\n# Pulling the season stats\nprint(\'Pulling the season stats...\')\nstats_df_2012_13 = LeagueDashPlayerStats(                ### edit year\n    season="2012-13",                                    ### edit year\n    season_type_all_star="Regular Season",\n    per_mode_detailed="PerGame"\n).get_data_frames()[0]\n\n\nprint(\'Recording Common Player Info...\')\nplayer_ids = stats_df_2012_13["PLAYER_ID"].unique()     ### edit year\n\nexperience_data = []\n\nfor pid in player_ids:\n    try:\n        info = CommonPlayerInfo(player_id=pid)\n        df = info.get_data_frames()[0]\n        exp = df["DRAFT_NUMBER"].iloc[0]\n        experience_data.append((pid, exp))\n        time.sleep(0.4)    # ✅ Rate-limit protection\n    except Exception as e:\n        print(f"Failed on {pid}: {e}")\n        experience_data.append((pid, None))\n\n\nprint(\'Merging dataframes...\')\n\n# Creating the experience dataframe\nexp_df_2012_13 = pd.DataFrame(experience_data, columns=["PLAYER_ID

### cIV.) 2013-14 Season
##### Rookie of the Year: Michael Carter-Williams

In [15]:
# Commented this out so it runs smoothly for you
'''
time1 = time.time()
# Pulling the season stats
stats_df_2013_14 = LeagueDashPlayerStats(                ### edit year
    season="2013-14",                                    ### edit year
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame"
).get_data_frames()[0]

player_ids = stats_df_2013_14["PLAYER_ID"].unique()     ### edit year

experience_data = []

for pid in player_ids:
    try:
        info = CommonPlayerInfo(player_id=pid)
        df = info.get_data_frames()[0]
        exp = df["DRAFT_NUMBER"].iloc[0]
        experience_data.append((pid, exp))
        time.sleep(0.4)    # ✅ Rate-limit protection
    except Exception as e:
        print(f"Failed on {pid}: {e}")
        experience_data.append((pid, None))

exp_df_2013_14 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year

# Merge with stats
full_df_2013_14 = stats_df_2013_14.merge(exp_df_2013_14, on="PLAYER_ID", how="left")             ### edit year


time2 = time.time()
timediff = time2-time1
tdm = timediff/60
print('This took', round(tdm,3), 'minutes.')

# This is all the progress you need to make here.
# Commented this out so it runs smoothly for you
'''

'\ntime1 = time.time()\n# Pulling the season stats\nstats_df_2013_14 = LeagueDashPlayerStats(                ### edit year\n    season="2013-14",                                    ### edit year\n    season_type_all_star="Regular Season",\n    per_mode_detailed="PerGame"\n).get_data_frames()[0]\n\nplayer_ids = stats_df_2013_14["PLAYER_ID"].unique()     ### edit year\n\nexperience_data = []\n\nfor pid in player_ids:\n    try:\n        info = CommonPlayerInfo(player_id=pid)\n        df = info.get_data_frames()[0]\n        exp = df["DRAFT_NUMBER"].iloc[0]\n        experience_data.append((pid, exp))\n        time.sleep(0.4)    # ✅ Rate-limit protection\n    except Exception as e:\n        print(f"Failed on {pid}: {e}")\n        experience_data.append((pid, None))\n\nexp_df_2013_14 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year\n\n# Merge with stats\nfull_df_2013_14 = stats_df_2013_14.merge(exp_df_2013_14, on="PLAYER_ID", how="left")       

### cV.) 2014-15 Season
##### Rookie of the Year: Andrew Wiggins

In [16]:
# Commented this out so it runs smoothly for you
'''
time1 = time.time()
# Pulling the season stats
stats_df_2014_15 = LeagueDashPlayerStats(                ### edit year
    season="2014-15",                                    ### edit year
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame"
).get_data_frames()[0]

player_ids = stats_df_2014_15["PLAYER_ID"].unique()     ### edit year

experience_data = []

for pid in player_ids:
    try:
        info = CommonPlayerInfo(player_id=pid)
        df = info.get_data_frames()[0]
        exp = df["DRAFT_NUMBER"].iloc[0]
        experience_data.append((pid, exp))
        time.sleep(0.4)    # ✅ Rate-limit protection
    except Exception as e:
        print(f"Failed on {pid}: {e}")
        experience_data.append((pid, None))

exp_df_2014_15 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year

# Merge with stats
full_df_2014_15 = stats_df_2014_15.merge(exp_df_2014_15, on="PLAYER_ID", how="left")             ### edit year


time2 = time.time()
timediff = time2-time1
tdm = timediff/60
print('This took', round(tdm,3), 'minutes to run.')

# This is all the progress you need to make here.
# Commented this out so it runs smoothly for you
'''

'\ntime1 = time.time()\n# Pulling the season stats\nstats_df_2014_15 = LeagueDashPlayerStats(                ### edit year\n    season="2014-15",                                    ### edit year\n    season_type_all_star="Regular Season",\n    per_mode_detailed="PerGame"\n).get_data_frames()[0]\n\nplayer_ids = stats_df_2014_15["PLAYER_ID"].unique()     ### edit year\n\nexperience_data = []\n\nfor pid in player_ids:\n    try:\n        info = CommonPlayerInfo(player_id=pid)\n        df = info.get_data_frames()[0]\n        exp = df["DRAFT_NUMBER"].iloc[0]\n        experience_data.append((pid, exp))\n        time.sleep(0.4)    # ✅ Rate-limit protection\n    except Exception as e:\n        print(f"Failed on {pid}: {e}")\n        experience_data.append((pid, None))\n\nexp_df_2014_15 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year\n\n# Merge with stats\nfull_df_2014_15 = stats_df_2014_15.merge(exp_df_2014_15, on="PLAYER_ID", how="left")       

### cVI.) 2015-16 Season
##### Rookie of the Year: Karl-Anthony Towns

In [17]:
# Commented this out so it runs smoothly for you
'''
time1 = time.time()
# Pulling the season stats
stats_df_2015_16 = LeagueDashPlayerStats(                ### edit year
    season="2015-16",                                    ### edit year
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame"
).get_data_frames()[0]

player_ids = stats_df_2015_16["PLAYER_ID"].unique()     ### edit year

experience_data = []

for pid in player_ids:
    try:
        info = CommonPlayerInfo(player_id=pid)
        df = info.get_data_frames()[0]
        exp = df["DRAFT_NUMBER"].iloc[0]
        experience_data.append((pid, exp))
        time.sleep(0.4)    # ✅ Rate-limit protection
    except Exception as e:
        print(f"Failed on {pid}: {e}")
        experience_data.append((pid, None))

exp_df_2015_16 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year

# Merge with stats
full_df_2015_16 = stats_df_2015_16.merge(exp_df_2015_16, on="PLAYER_ID", how="left")             ### edit year


time2 = time.time()
timediff = time2-time1
tdm = timediff/60
print('This took', round(tdm,3), 'minutes to run.')

# This is all the progress you need to make here.
# Commented this out so it runs smoothly for you
'''

'\ntime1 = time.time()\n# Pulling the season stats\nstats_df_2015_16 = LeagueDashPlayerStats(                ### edit year\n    season="2015-16",                                    ### edit year\n    season_type_all_star="Regular Season",\n    per_mode_detailed="PerGame"\n).get_data_frames()[0]\n\nplayer_ids = stats_df_2015_16["PLAYER_ID"].unique()     ### edit year\n\nexperience_data = []\n\nfor pid in player_ids:\n    try:\n        info = CommonPlayerInfo(player_id=pid)\n        df = info.get_data_frames()[0]\n        exp = df["DRAFT_NUMBER"].iloc[0]\n        experience_data.append((pid, exp))\n        time.sleep(0.4)    # ✅ Rate-limit protection\n    except Exception as e:\n        print(f"Failed on {pid}: {e}")\n        experience_data.append((pid, None))\n\nexp_df_2015_16 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year\n\n# Merge with stats\nfull_df_2015_16 = stats_df_2015_16.merge(exp_df_2015_16, on="PLAYER_ID", how="left")       

### cVII.) 2016-17 Season
##### Rookie of the Year: Malcolm Brogdon

In [18]:
# Commented this out so it runs smoothly for you
'''
time1 = time.time()
# Pulling the season stats
stats_df_2016_17 = LeagueDashPlayerStats(                ### edit year
    season="2016-17",                                    ### edit year
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame"
).get_data_frames()[0]

player_ids = stats_df_2016_17["PLAYER_ID"].unique()     ### edit year

experience_data = []

for pid in player_ids:
    try:
        info = CommonPlayerInfo(player_id=pid)
        df = info.get_data_frames()[0]
        exp = df["DRAFT_NUMBER"].iloc[0]
        experience_data.append((pid, exp))
        time.sleep(0.4)    # ✅ Rate-limit protection
    except Exception as e:
        print(f"Failed on {pid}: {e}")
        experience_data.append((pid, None))

exp_df_2016_17 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year

# Merge with stats
full_df_2016_17 = stats_df_2016_17.merge(exp_df_2016_17, on="PLAYER_ID", how="left")             ### edit year


time2 = time.time()
timediff = time2-time1
tdm = timediff/60
print('This took', round(tdm,3), 'minutes to run.')

# This is all the progress you need to make here.
# Commented this out so it runs smoothly for you
'''

'\ntime1 = time.time()\n# Pulling the season stats\nstats_df_2016_17 = LeagueDashPlayerStats(                ### edit year\n    season="2016-17",                                    ### edit year\n    season_type_all_star="Regular Season",\n    per_mode_detailed="PerGame"\n).get_data_frames()[0]\n\nplayer_ids = stats_df_2016_17["PLAYER_ID"].unique()     ### edit year\n\nexperience_data = []\n\nfor pid in player_ids:\n    try:\n        info = CommonPlayerInfo(player_id=pid)\n        df = info.get_data_frames()[0]\n        exp = df["DRAFT_NUMBER"].iloc[0]\n        experience_data.append((pid, exp))\n        time.sleep(0.4)    # ✅ Rate-limit protection\n    except Exception as e:\n        print(f"Failed on {pid}: {e}")\n        experience_data.append((pid, None))\n\nexp_df_2016_17 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year\n\n# Merge with stats\nfull_df_2016_17 = stats_df_2016_17.merge(exp_df_2016_17, on="PLAYER_ID", how="left")       

### cVIII.) 2017-18 Season
##### Rookie of the Year: Ben Simmons

In [19]:
# Commented this out so it runs smoothly for you
'''
time1 = time.time()
# Pulling the season stats
stats_df_2017_18 = LeagueDashPlayerStats(                ### edit year
    season="2017-18",                                    ### edit year
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame"
).get_data_frames()[0]

player_ids = stats_df_2017_18["PLAYER_ID"].unique()     ### edit year

experience_data = []

for pid in player_ids:
    try:
        info = CommonPlayerInfo(player_id=pid)
        df = info.get_data_frames()[0]
        exp = df["DRAFT_NUMBER"].iloc[0]
        experience_data.append((pid, exp))
        time.sleep(0.4)    # ✅ Rate-limit protection
    except Exception as e:
        print(f"Failed on {pid}: {e}")
        experience_data.append((pid, None))

exp_df_2017_18 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year

# Merge with stats
full_df_2017_18 = stats_df_2017_18.merge(exp_df_2017_18, on="PLAYER_ID", how="left")             ### edit year


time2 = time.time()
timediff = time2-time1
tdm = timediff/60
print('This took', round(tdm,3), 'minutes to run.')

# This is all the progress you need to make here.
# Commented this out so it runs smoothly for you
'''

'\ntime1 = time.time()\n# Pulling the season stats\nstats_df_2017_18 = LeagueDashPlayerStats(                ### edit year\n    season="2017-18",                                    ### edit year\n    season_type_all_star="Regular Season",\n    per_mode_detailed="PerGame"\n).get_data_frames()[0]\n\nplayer_ids = stats_df_2017_18["PLAYER_ID"].unique()     ### edit year\n\nexperience_data = []\n\nfor pid in player_ids:\n    try:\n        info = CommonPlayerInfo(player_id=pid)\n        df = info.get_data_frames()[0]\n        exp = df["DRAFT_NUMBER"].iloc[0]\n        experience_data.append((pid, exp))\n        time.sleep(0.4)    # ✅ Rate-limit protection\n    except Exception as e:\n        print(f"Failed on {pid}: {e}")\n        experience_data.append((pid, None))\n\nexp_df_2017_18 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year\n\n# Merge with stats\nfull_df_2017_18 = stats_df_2017_18.merge(exp_df_2017_18, on="PLAYER_ID", how="left")       

### cIX.) 2018-19 Season
##### Rookie of the Year: Luka Dončić

In [20]:
# Commented this out so it runs smoothly for you
'''
time1 = time.time()
# Pulling the season stats
stats_df_2018_19 = LeagueDashPlayerStats(                ### edit year
    season="2018-19",                                    ### edit year
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame"
).get_data_frames()[0]

player_ids = stats_df_2018_19["PLAYER_ID"].unique()     ### edit year

experience_data = []

for pid in player_ids:
    try:
        info = CommonPlayerInfo(player_id=pid)
        df = info.get_data_frames()[0]
        exp = df["DRAFT_NUMBER"].iloc[0]
        experience_data.append((pid, exp))
        time.sleep(0.4)    # ✅ Rate-limit protection
    except Exception as e:
        print(f"Failed on {pid}: {e}")
        experience_data.append((pid, None))

exp_df_2018_19 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year

# Merge with stats
full_df_2018_19 = stats_df_2018_19.merge(exp_df_2018_19, on="PLAYER_ID", how="left")             ### edit year


time2 = time.time()
timediff = time2-time1
tdm = timediff/60
print('This took', round(tdm,3), 'minutes to run.')

# This is all the progress you need to make here.
# Commented this out so it runs smoothly for you
'''

'\ntime1 = time.time()\n# Pulling the season stats\nstats_df_2018_19 = LeagueDashPlayerStats(                ### edit year\n    season="2018-19",                                    ### edit year\n    season_type_all_star="Regular Season",\n    per_mode_detailed="PerGame"\n).get_data_frames()[0]\n\nplayer_ids = stats_df_2018_19["PLAYER_ID"].unique()     ### edit year\n\nexperience_data = []\n\nfor pid in player_ids:\n    try:\n        info = CommonPlayerInfo(player_id=pid)\n        df = info.get_data_frames()[0]\n        exp = df["DRAFT_NUMBER"].iloc[0]\n        experience_data.append((pid, exp))\n        time.sleep(0.4)    # ✅ Rate-limit protection\n    except Exception as e:\n        print(f"Failed on {pid}: {e}")\n        experience_data.append((pid, None))\n\nexp_df_2018_19 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year\n\n# Merge with stats\nfull_df_2018_19 = stats_df_2018_19.merge(exp_df_2018_19, on="PLAYER_ID", how="left")       

### cX.) 2019-20 Season
##### Rookie of the Year: Ja Morant

In [21]:
# Commented this out so it runs smoothly for you
'''
time1 = time.time()
# Pulling the season stats
stats_df_2019_20 = LeagueDashPlayerStats(                ### edit year
    season="2019-20",                                    ### edit year
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame"
).get_data_frames()[0]

player_ids = stats_df_2019_20["PLAYER_ID"].unique()     ### edit year

experience_data = []

for pid in player_ids:
    try:
        info = CommonPlayerInfo(player_id=pid)
        df = info.get_data_frames()[0]
        exp = df["DRAFT_NUMBER"].iloc[0]
        experience_data.append((pid, exp))
        time.sleep(0.4)    # ✅ Rate-limit protection
    except Exception as e:
        print(f"Failed on {pid}: {e}")
        experience_data.append((pid, None))

exp_df_2019_20 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year

# Merge with stats
full_df_2019_20 = stats_df_2019_20.merge(exp_df_2019_20, on="PLAYER_ID", how="left")             ### edit year


time2 = time.time()
timediff = time2-time1
tdm = timediff/60
print('This took', round(tdm,3), 'minutes to run.')

# This is all the progress you need to make here.
# Commented this out so it runs smoothly for you
'''

'\ntime1 = time.time()\n# Pulling the season stats\nstats_df_2019_20 = LeagueDashPlayerStats(                ### edit year\n    season="2019-20",                                    ### edit year\n    season_type_all_star="Regular Season",\n    per_mode_detailed="PerGame"\n).get_data_frames()[0]\n\nplayer_ids = stats_df_2019_20["PLAYER_ID"].unique()     ### edit year\n\nexperience_data = []\n\nfor pid in player_ids:\n    try:\n        info = CommonPlayerInfo(player_id=pid)\n        df = info.get_data_frames()[0]\n        exp = df["DRAFT_NUMBER"].iloc[0]\n        experience_data.append((pid, exp))\n        time.sleep(0.4)    # ✅ Rate-limit protection\n    except Exception as e:\n        print(f"Failed on {pid}: {e}")\n        experience_data.append((pid, None))\n\nexp_df_2019_20 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year\n\n# Merge with stats\nfull_df_2019_20 = stats_df_2019_20.merge(exp_df_2019_20, on="PLAYER_ID", how="left")       

### cXI.) 2020-21 Season
##### Rookie of the Year: LaMelo Ball

In [22]:
# Commented this out so it runs smoothly for you
'''
time1 = time.time()
# Pulling the season stats
stats_df_2020_21 = LeagueDashPlayerStats(                ### edit year
    season="2020-21",                                    ### edit year
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame"
).get_data_frames()[0]

player_ids = stats_df_2020_21["PLAYER_ID"].unique()     ### edit year

experience_data = []

for pid in player_ids:
    try:
        info = CommonPlayerInfo(player_id=pid)
        df = info.get_data_frames()[0]
        exp = df["DRAFT_NUMBER"].iloc[0]
        experience_data.append((pid, exp))
        time.sleep(0.4)    # ✅ Rate-limit protection
    except Exception as e:
        print(f"Failed on {pid}: {e}")
        experience_data.append((pid, None))

exp_df_2020_21 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year

# Merge with stats
full_df_2020_21 = stats_df_2020_21.merge(exp_df_2020_21, on="PLAYER_ID", how="left")             ### edit year


time2 = time.time()
timediff = time2-time1
tdm = timediff/60
print('This took', round(tdm,3), 'minutes to run.')

# This is all the progress you need to make here.
# Commented this out so it runs smoothly for you
'''

'\ntime1 = time.time()\n# Pulling the season stats\nstats_df_2020_21 = LeagueDashPlayerStats(                ### edit year\n    season="2020-21",                                    ### edit year\n    season_type_all_star="Regular Season",\n    per_mode_detailed="PerGame"\n).get_data_frames()[0]\n\nplayer_ids = stats_df_2020_21["PLAYER_ID"].unique()     ### edit year\n\nexperience_data = []\n\nfor pid in player_ids:\n    try:\n        info = CommonPlayerInfo(player_id=pid)\n        df = info.get_data_frames()[0]\n        exp = df["DRAFT_NUMBER"].iloc[0]\n        experience_data.append((pid, exp))\n        time.sleep(0.4)    # ✅ Rate-limit protection\n    except Exception as e:\n        print(f"Failed on {pid}: {e}")\n        experience_data.append((pid, None))\n\nexp_df_2020_21 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year\n\n# Merge with stats\nfull_df_2020_21 = stats_df_2020_21.merge(exp_df_2020_21, on="PLAYER_ID", how="left")       

### cXII.) 2021-22 Season
##### Rookie of the Year: Scottie Barnes

In [23]:
# Commented this out so it runs smoothly for you
'''
time1 = time.time()
# Pulling the season stats
stats_df_2021_22 = LeagueDashPlayerStats(                ### edit year
    season="2021-22",                                    ### edit year
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame"
).get_data_frames()[0]

player_ids = stats_df_2021_22["PLAYER_ID"].unique()     ### edit year

experience_data = []

for pid in player_ids:
    try:
        info = CommonPlayerInfo(player_id=pid)
        df = info.get_data_frames()[0]
        exp = df["DRAFT_NUMBER"].iloc[0]
        experience_data.append((pid, exp))
        time.sleep(0.4)    # ✅ Rate-limit protection
    except Exception as e:
        print(f"Failed on {pid}: {e}")
        experience_data.append((pid, None))

exp_df_2021_22 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year

# Merge with stats
full_df_2021_22 = stats_df_2021_22.merge(exp_df_2021_22, on="PLAYER_ID", how="left")             ### edit year


time2 = time.time()
timediff = time2-time1
tdm = timediff/60
print('This took', round(tdm,3), 'minutes to run.')

# This is all the progress you need to make here.
# Commented this out so it runs smoothly for you
'''

'\ntime1 = time.time()\n# Pulling the season stats\nstats_df_2021_22 = LeagueDashPlayerStats(                ### edit year\n    season="2021-22",                                    ### edit year\n    season_type_all_star="Regular Season",\n    per_mode_detailed="PerGame"\n).get_data_frames()[0]\n\nplayer_ids = stats_df_2021_22["PLAYER_ID"].unique()     ### edit year\n\nexperience_data = []\n\nfor pid in player_ids:\n    try:\n        info = CommonPlayerInfo(player_id=pid)\n        df = info.get_data_frames()[0]\n        exp = df["DRAFT_NUMBER"].iloc[0]\n        experience_data.append((pid, exp))\n        time.sleep(0.4)    # ✅ Rate-limit protection\n    except Exception as e:\n        print(f"Failed on {pid}: {e}")\n        experience_data.append((pid, None))\n\nexp_df_2021_22 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year\n\n# Merge with stats\nfull_df_2021_22 = stats_df_2021_22.merge(exp_df_2021_22, on="PLAYER_ID", how="left")       

### cXIII.) 2022-23 Season
##### Rookie of the Year: Paolo Banchero

In [24]:
# Commented this out so it runs smoothly for you
'''
time1 = time.time()
# Pulling the season stats
stats_df_2022_23 = LeagueDashPlayerStats(                ### edit year
    season="2022-23",                                    ### edit year
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame"
).get_data_frames()[0]

player_ids = stats_df_2022_23["PLAYER_ID"].unique()     ### edit year

experience_data = []

for pid in player_ids:
    try:
        info = CommonPlayerInfo(player_id=pid)
        df = info.get_data_frames()[0]
        exp = df["DRAFT_NUMBER"].iloc[0]
        experience_data.append((pid, exp))
        time.sleep(0.4)    # ✅ Rate-limit protection
    except Exception as e:
        print(f"Failed on {pid}: {e}")
        experience_data.append((pid, None))

exp_df_2022_23 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year

# Merge with stats
full_df_2022_23 = stats_df_2022_23.merge(exp_df_2022_23, on="PLAYER_ID", how="left")             ### edit year


time2 = time.time()
timediff = time2-time1
tdm = timediff/60
print('This took', round(tdm,3), 'minutes to run.')

# This is all the progress you need to make here.
# Commented this out so it runs smoothly for you
'''

'\ntime1 = time.time()\n# Pulling the season stats\nstats_df_2022_23 = LeagueDashPlayerStats(                ### edit year\n    season="2022-23",                                    ### edit year\n    season_type_all_star="Regular Season",\n    per_mode_detailed="PerGame"\n).get_data_frames()[0]\n\nplayer_ids = stats_df_2022_23["PLAYER_ID"].unique()     ### edit year\n\nexperience_data = []\n\nfor pid in player_ids:\n    try:\n        info = CommonPlayerInfo(player_id=pid)\n        df = info.get_data_frames()[0]\n        exp = df["DRAFT_NUMBER"].iloc[0]\n        experience_data.append((pid, exp))\n        time.sleep(0.4)    # ✅ Rate-limit protection\n    except Exception as e:\n        print(f"Failed on {pid}: {e}")\n        experience_data.append((pid, None))\n\nexp_df_2022_23 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year\n\n# Merge with stats\nfull_df_2022_23 = stats_df_2022_23.merge(exp_df_2022_23, on="PLAYER_ID", how="left")       

### cXIV.) 2023-24 Season
##### Rookie of the Year: Victor Wembanyama

In [25]:
# Commented this out so it runs smoothly for you
'''
time1 = time.time()
# Pulling the season stats
stats_df_2023_24 = LeagueDashPlayerStats(                ### edit year
    season="2023-24",                                    ### edit year
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame"
).get_data_frames()[0]

player_ids = stats_df_2023_24["PLAYER_ID"].unique()     ### edit year

experience_data = []

for pid in player_ids:
    try:
        info = CommonPlayerInfo(player_id=pid)
        df = info.get_data_frames()[0]
        exp = df["DRAFT_NUMBER"].iloc[0]
        experience_data.append((pid, exp))
        time.sleep(0.4)    # ✅ Rate-limit protection
    except Exception as e:
        print(f"Failed on {pid}: {e}")
        experience_data.append((pid, None))

exp_df_2023_24 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year

# Merge with stats
full_df_2023_24 = stats_df_2023_24.merge(exp_df_2023_24, on="PLAYER_ID", how="left")             ### edit year


time2 = time.time()
timediff = time2-time1
tdm = timediff/60
print('This took', round(tdm,3), 'minutes to run.')

# This is all the progress you need to make here.
# Commented this out so it runs smoothly for you
'''

'\ntime1 = time.time()\n# Pulling the season stats\nstats_df_2023_24 = LeagueDashPlayerStats(                ### edit year\n    season="2023-24",                                    ### edit year\n    season_type_all_star="Regular Season",\n    per_mode_detailed="PerGame"\n).get_data_frames()[0]\n\nplayer_ids = stats_df_2023_24["PLAYER_ID"].unique()     ### edit year\n\nexperience_data = []\n\nfor pid in player_ids:\n    try:\n        info = CommonPlayerInfo(player_id=pid)\n        df = info.get_data_frames()[0]\n        exp = df["DRAFT_NUMBER"].iloc[0]\n        experience_data.append((pid, exp))\n        time.sleep(0.4)    # ✅ Rate-limit protection\n    except Exception as e:\n        print(f"Failed on {pid}: {e}")\n        experience_data.append((pid, None))\n\nexp_df_2023_24 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year\n\n# Merge with stats\nfull_df_2023_24 = stats_df_2023_24.merge(exp_df_2023_24, on="PLAYER_ID", how="left")       

### cXV.) 2024-25 Season
##### Rookie of the Year: Stephon Castle

In [26]:
# Commented this out so it runs smoothly for you
'''
time1 = time.time()
# Pulling the season stats
stats_df_2024_25 = LeagueDashPlayerStats(                ### edit year
    season="2024-25",                                    ### edit year
    season_type_all_star="Regular Season",
    per_mode_detailed="PerGame"
).get_data_frames()[0]

player_ids = stats_df_2024_25["PLAYER_ID"].unique()     ### edit year

experience_data = []

for pid in player_ids:
    try:
        info = CommonPlayerInfo(player_id=pid)
        df = info.get_data_frames()[0]
        exp = df["DRAFT_NUMBER"].iloc[0]
        experience_data.append((pid, exp))
        time.sleep(0.4)    # ✅ Rate-limit protection
    except Exception as e:
        print(f"Failed on {pid}: {e}")
        experience_data.append((pid, None))

exp_df_2024_25 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year

# Merge with stats
full_df_2024_25 = stats_df_2024_25.merge(exp_df_2024_25, on="PLAYER_ID", how="left")             ### edit year


time2 = time.time()
timediff = time2-time1
tdm = timediff/60
print('This took', round(tdm,3), 'minutes to run.')
# Commented this out so it runs smoothly for you
'''

'\ntime1 = time.time()\n# Pulling the season stats\nstats_df_2024_25 = LeagueDashPlayerStats(                ### edit year\n    season="2024-25",                                    ### edit year\n    season_type_all_star="Regular Season",\n    per_mode_detailed="PerGame"\n).get_data_frames()[0]\n\nplayer_ids = stats_df_2024_25["PLAYER_ID"].unique()     ### edit year\n\nexperience_data = []\n\nfor pid in player_ids:\n    try:\n        info = CommonPlayerInfo(player_id=pid)\n        df = info.get_data_frames()[0]\n        exp = df["DRAFT_NUMBER"].iloc[0]\n        experience_data.append((pid, exp))\n        time.sleep(0.4)    # ✅ Rate-limit protection\n    except Exception as e:\n        print(f"Failed on {pid}: {e}")\n        experience_data.append((pid, None))\n\nexp_df_2024_25 = pd.DataFrame(experience_data, columns=["PLAYER_ID", "DRAFT_NUMBER"])              ### edit year\n\n# Merge with stats\nfull_df_2024_25 = stats_df_2024_25.merge(exp_df_2024_25, on="PLAYER_ID", how="left")       

### cXVI.) Calculating PRA, GP_pct Historical Data

In [27]:
# Commented this out so it runs smoothly for you
'''
# PRA calculation
full_df_2024_25["PRA"] = full_df_2024_25["PTS"] + full_df_2024_25["REB"] + full_df_2024_25["AST"]
full_df_2023_24["PRA"] = full_df_2023_24["PTS"] + full_df_2023_24["REB"] + full_df_2023_24["AST"]
full_df_2022_23["PRA"] = full_df_2022_23["PTS"] + full_df_2022_23["REB"] + full_df_2022_23["AST"]
full_df_2021_22["PRA"] = full_df_2021_22["PTS"] + full_df_2021_22["REB"] + full_df_2021_22["AST"]
full_df_2020_21["PRA"] = full_df_2020_21["PTS"] + full_df_2020_21["REB"] + full_df_2020_21["AST"]
full_df_2019_20["PRA"] = full_df_2019_20["PTS"] + full_df_2019_20["REB"] + full_df_2019_20["AST"]
full_df_2018_19["PRA"] = full_df_2018_19["PTS"] + full_df_2018_19["REB"] + full_df_2018_19["AST"]
full_df_2017_18["PRA"] = full_df_2017_18["PTS"] + full_df_2017_18["REB"] + full_df_2017_18["AST"]
full_df_2016_17["PRA"] = full_df_2016_17["PTS"] + full_df_2016_17["REB"] + full_df_2016_17["AST"]
full_df_2015_16["PRA"] = full_df_2015_16["PTS"] + full_df_2015_16["REB"] + full_df_2015_16["AST"]
full_df_2014_15["PRA"] = full_df_2014_15["PTS"] + full_df_2014_15["REB"] + full_df_2014_15["AST"]
full_df_2013_14["PRA"] = full_df_2013_14["PTS"] + full_df_2013_14["REB"] + full_df_2013_14["AST"]
full_df_2012_13["PRA"] = full_df_2012_13["PTS"] + full_df_2012_13["REB"] + full_df_2012_13["AST"]
full_df_2011_12["PRA"] = full_df_2011_12["PTS"] + full_df_2011_12["REB"] + full_df_2011_12["AST"]
full_df_2010_11["PRA"] = full_df_2010_11["PTS"] + full_df_2010_11["REB"] + full_df_2010_11["AST"]

# Games in the season for teams
full_df_2024_25["TOT_GAMES"] = 82
full_df_2023_24["TOT_GAMES"] = 82
full_df_2022_23["TOT_GAMES"] = 82
full_df_2021_22["TOT_GAMES"] = 82
full_df_2020_21["TOT_GAMES"] = 72
full_df_2019_20["TOT_GAMES"] = 0
full_df_2018_19["TOT_GAMES"] = 82
full_df_2017_18["TOT_GAMES"] = 82
full_df_2016_17["TOT_GAMES"] = 82
full_df_2015_16["TOT_GAMES"] = 82
full_df_2014_15["TOT_GAMES"] = 82
full_df_2013_14["TOT_GAMES"] = 82
full_df_2012_13["TOT_GAMES"] = 82
full_df_2011_12["TOT_GAMES"] = 66
full_df_2010_11["TOT_GAMES"] = 82

# Correcting the 2019-20 games played (source: wikipedia https://en.wikipedia.org/wiki/2019%E2%80%9320_NBA_season)
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="MIL", "TOT_GAMES"] = 73
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="WAS", "TOT_GAMES"] = 72
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="DEN", "TOT_GAMES"] = 73
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="HOU", "TOT_GAMES"] = 72
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="IND", "TOT_GAMES"] = 73
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="OKC", "TOT_GAMES"] = 72
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="UTA", "TOT_GAMES"] = 72
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="PHI", "TOT_GAMES"] = 73
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="LAL", "TOT_GAMES"] = 71
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="GSW", "TOT_GAMES"] = 65
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="BOS", "TOT_GAMES"] = 72
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="MIA", "TOT_GAMES"] = 73
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="ORL", "TOT_GAMES"] = 73
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="DAL", "TOT_GAMES"] = 75
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="MIN", "TOT_GAMES"] = 64
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="CHA", "TOT_GAMES"] = 65
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="NYK", "TOT_GAMES"] = 66
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="ATL", "TOT_GAMES"] = 67
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="DET", "TOT_GAMES"] = 66
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="CHI", "TOT_GAMES"] = 65
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="BKN", "TOT_GAMES"] = 72
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="SAS", "TOT_GAMES"] = 71
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="POR", "TOT_GAMES"] = 74
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="LAC", "TOT_GAMES"] = 72
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="TOR", "TOT_GAMES"] = 72
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="NOP", "TOT_GAMES"] = 72
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="MEM", "TOT_GAMES"] = 73
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="CLE", "TOT_GAMES"] = 65
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="PHX", "TOT_GAMES"] = 73
full_df_2019_20.loc[full_df_2019_20["TEAM_ABBREVIATION"]=="SAC", "TOT_GAMES"] = 72


# % Games played by player
full_df_2024_25["GP_pct"] = full_df_2024_25["GP"] / full_df_2024_25["TOT_GAMES"]
full_df_2023_24["GP_pct"] = full_df_2023_24["GP"] / full_df_2023_24["TOT_GAMES"]
full_df_2022_23["GP_pct"] = full_df_2022_23["GP"] / full_df_2022_23["TOT_GAMES"]
full_df_2021_22["GP_pct"] = full_df_2021_22["GP"] / full_df_2021_22["TOT_GAMES"]
full_df_2020_21["GP_pct"] = full_df_2020_21["GP"] / full_df_2020_21["TOT_GAMES"]
full_df_2019_20["GP_pct"] = full_df_2019_20["GP"] / full_df_2019_20["TOT_GAMES"]
full_df_2018_19["GP_pct"] = full_df_2018_19["GP"] / full_df_2018_19["TOT_GAMES"]
full_df_2017_18["GP_pct"] = full_df_2017_18["GP"] / full_df_2017_18["TOT_GAMES"]
full_df_2016_17["GP_pct"] = full_df_2016_17["GP"] / full_df_2016_17["TOT_GAMES"]
full_df_2015_16["GP_pct"] = full_df_2015_16["GP"] / full_df_2015_16["TOT_GAMES"]
full_df_2014_15["GP_pct"] = full_df_2014_15["GP"] / full_df_2014_15["TOT_GAMES"]
full_df_2013_14["GP_pct"] = full_df_2013_14["GP"] / full_df_2013_14["TOT_GAMES"]
full_df_2012_13["GP_pct"] = full_df_2012_13["GP"] / full_df_2012_13["TOT_GAMES"]
full_df_2011_12["GP_pct"] = full_df_2011_12["GP"] / full_df_2011_12["TOT_GAMES"]
full_df_2010_11["GP_pct"] = full_df_2010_11["GP"] / full_df_2010_11["TOT_GAMES"]

# Calculating Opportunity Metric
full_df_2024_25["Opp_met"] = (full_df_2024_25["MIN"]) * (full_df_2024_25["GP_pct"]) * (1+((full_df_2024_25["PLUS_MINUS"])/10))
full_df_2023_24["Opp_met"] = (full_df_2023_24["MIN"]) * (full_df_2023_24["GP_pct"]) * (1+((full_df_2023_24["PLUS_MINUS"])/10))
full_df_2022_23["Opp_met"] = (full_df_2022_23["MIN"]) * (full_df_2022_23["GP_pct"]) * (1+((full_df_2022_23["PLUS_MINUS"])/10))
full_df_2021_22["Opp_met"] = (full_df_2021_22["MIN"]) * (full_df_2021_22["GP_pct"]) * (1+((full_df_2021_22["PLUS_MINUS"])/10))
full_df_2020_21["Opp_met"] = (full_df_2020_21["MIN"]) * (full_df_2020_21["GP_pct"]) * (1+((full_df_2020_21["PLUS_MINUS"])/10))
full_df_2019_20["Opp_met"] = (full_df_2019_20["MIN"]) * (full_df_2019_20["GP_pct"]) * (1+((full_df_2019_20["PLUS_MINUS"])/10))
full_df_2018_19["Opp_met"] = (full_df_2018_19["MIN"]) * (full_df_2018_19["GP_pct"]) * (1+((full_df_2018_19["PLUS_MINUS"])/10))
full_df_2017_18["Opp_met"] = (full_df_2017_18["MIN"]) * (full_df_2017_18["GP_pct"]) * (1+((full_df_2017_18["PLUS_MINUS"])/10))
full_df_2016_17["Opp_met"] = (full_df_2016_17["MIN"]) * (full_df_2016_17["GP_pct"]) * (1+((full_df_2016_17["PLUS_MINUS"])/10))
full_df_2015_16["Opp_met"] = (full_df_2015_16["MIN"]) * (full_df_2015_16["GP_pct"]) * (1+((full_df_2015_16["PLUS_MINUS"])/10))
full_df_2014_15["Opp_met"] = (full_df_2014_15["MIN"]) * (full_df_2014_15["GP_pct"]) * (1+((full_df_2014_15["PLUS_MINUS"])/10))
full_df_2013_14["Opp_met"] = (full_df_2013_14["MIN"]) * (full_df_2013_14["GP_pct"]) * (1+((full_df_2013_14["PLUS_MINUS"])/10))
full_df_2012_13["Opp_met"] = (full_df_2012_13["MIN"]) * (full_df_2012_13["GP_pct"]) * (1+((full_df_2012_13["PLUS_MINUS"])/10))
full_df_2011_12["Opp_met"] = (full_df_2011_12["MIN"]) * (full_df_2011_12["GP_pct"]) * (1+((full_df_2011_12["PLUS_MINUS"])/10))
full_df_2010_11["Opp_met"] = (full_df_2010_11["MIN"]) * (full_df_2010_11["GP_pct"]) * (1+((full_df_2010_11["PLUS_MINUS"])/10))
# Commented this out so it runs smoothly for you
'''

'\n# PRA calculation\nfull_df_2024_25["PRA"] = full_df_2024_25["PTS"] + full_df_2024_25["REB"] + full_df_2024_25["AST"]\nfull_df_2023_24["PRA"] = full_df_2023_24["PTS"] + full_df_2023_24["REB"] + full_df_2023_24["AST"]\nfull_df_2022_23["PRA"] = full_df_2022_23["PTS"] + full_df_2022_23["REB"] + full_df_2022_23["AST"]\nfull_df_2021_22["PRA"] = full_df_2021_22["PTS"] + full_df_2021_22["REB"] + full_df_2021_22["AST"]\nfull_df_2020_21["PRA"] = full_df_2020_21["PTS"] + full_df_2020_21["REB"] + full_df_2020_21["AST"]\nfull_df_2019_20["PRA"] = full_df_2019_20["PTS"] + full_df_2019_20["REB"] + full_df_2019_20["AST"]\nfull_df_2018_19["PRA"] = full_df_2018_19["PTS"] + full_df_2018_19["REB"] + full_df_2018_19["AST"]\nfull_df_2017_18["PRA"] = full_df_2017_18["PTS"] + full_df_2017_18["REB"] + full_df_2017_18["AST"]\nfull_df_2016_17["PRA"] = full_df_2016_17["PTS"] + full_df_2016_17["REB"] + full_df_2016_17["AST"]\nfull_df_2015_16["PRA"] = full_df_2015_16["PTS"] + full_df_2015_16["REB"] + full_df_2015

### cXVII.)  Adding the Season to each DF and saving all as CSVs

In [28]:
# Commented this out so it runs smoothly for you
'''
full_df_2010_11["SEASON"] = 2010
full_df_2011_12["SEASON"] = 2011
full_df_2012_13["SEASON"] = 2012
full_df_2013_14["SEASON"] = 2013
full_df_2014_15["SEASON"] = 2014
full_df_2015_16["SEASON"] = 2015
full_df_2016_17["SEASON"] = 2016
full_df_2017_18["SEASON"] = 2017
full_df_2018_19["SEASON"] = 2018
full_df_2019_20["SEASON"] = 2019
full_df_2020_21["SEASON"] = 2020
full_df_2021_22["SEASON"] = 2021
full_df_2022_23["SEASON"] = 2022
full_df_2023_24["SEASON"] = 2023
full_df_2024_25["SEASON"] = 2024
# Commented this out so it runs smoothly for you
'''

'\nfull_df_2010_11["SEASON"] = 2010\nfull_df_2011_12["SEASON"] = 2011\nfull_df_2012_13["SEASON"] = 2012\nfull_df_2013_14["SEASON"] = 2013\nfull_df_2014_15["SEASON"] = 2014\nfull_df_2015_16["SEASON"] = 2015\nfull_df_2016_17["SEASON"] = 2016\nfull_df_2017_18["SEASON"] = 2017\nfull_df_2018_19["SEASON"] = 2018\nfull_df_2019_20["SEASON"] = 2019\nfull_df_2020_21["SEASON"] = 2020\nfull_df_2021_22["SEASON"] = 2021\nfull_df_2022_23["SEASON"] = 2022\nfull_df_2023_24["SEASON"] = 2023\nfull_df_2024_25["SEASON"] = 2024\n# Commented this out so it runs smoothly for you\n'

In [29]:
# Commented this out so it runs smoothly for you
'''
full_df_2010_11.to_csv("season_2010_11_full.csv", index=False)
full_df_2011_12.to_csv("season_2011_12_full.csv", index=False)
full_df_2012_13.to_csv("season_2012_13_full.csv", index=False)
full_df_2013_14.to_csv("season_2013_14_full.csv", index=False)
full_df_2014_15.to_csv("season_2014_15_full.csv", index=False)
full_df_2015_16.to_csv("season_2015_16_full.csv", index=False)
full_df_2016_17.to_csv("season_2016_17_full.csv", index=False)
full_df_2017_18.to_csv("season_2017_18_full.csv", index=False)
full_df_2018_19.to_csv("season_2018_19_full.csv", index=False)
full_df_2019_20.to_csv("season_2019_20_full.csv", index=False)
full_df_2020_21.to_csv("season_2020_21_full.csv", index=False)
full_df_2021_22.to_csv("season_2021_22_full.csv", index=False)
full_df_2022_23.to_csv("season_2022_23_full.csv", index=False)
full_df_2023_24.to_csv("season_2023_24_full.csv", index=False)
full_df_2024_25.to_csv("season_2024_25_full.csv", index=False)
# Commented this out so it runs smoothly for you
'''

'\nfull_df_2010_11.to_csv("season_2010_11_full.csv", index=False)\nfull_df_2011_12.to_csv("season_2011_12_full.csv", index=False)\nfull_df_2012_13.to_csv("season_2012_13_full.csv", index=False)\nfull_df_2013_14.to_csv("season_2013_14_full.csv", index=False)\nfull_df_2014_15.to_csv("season_2014_15_full.csv", index=False)\nfull_df_2015_16.to_csv("season_2015_16_full.csv", index=False)\nfull_df_2016_17.to_csv("season_2016_17_full.csv", index=False)\nfull_df_2017_18.to_csv("season_2017_18_full.csv", index=False)\nfull_df_2018_19.to_csv("season_2018_19_full.csv", index=False)\nfull_df_2019_20.to_csv("season_2019_20_full.csv", index=False)\nfull_df_2020_21.to_csv("season_2020_21_full.csv", index=False)\nfull_df_2021_22.to_csv("season_2021_22_full.csv", index=False)\nfull_df_2022_23.to_csv("season_2022_23_full.csv", index=False)\nfull_df_2023_24.to_csv("season_2023_24_full.csv", index=False)\nfull_df_2024_25.to_csv("season_2024_25_full.csv", index=False)\n# Commented this out so it runs smoot

In [30]:
#ls

In [31]:
#print(full_df_2018_19["SEASON"])

### cXVIII.)  Saving all Dataframes into a Master CSV

In [32]:
#ls

In [33]:
# Commented this out so it runs smoothly for you
'''
listof_alldfs = ["season_2010_11_full.csv","season_2011_12_full.csv","season_2012_13_full.csv","season_2013_14_full.csv",
                 "season_2014_15_full.csv","season_2015_16_full.csv","season_2016_17_full.csv","season_2017_18_full.csv",
                 "season_2018_19_full.csv","season_2019_20_full.csv","season_2020_21_full.csv","season_2021_22_full.csv",
"season_2022_23_full.csv","season_2023_24_full.csv","season_2024_25_full.csv"]
dfs = [pd.read_csv(f) for f in listof_alldfs]

full_df = pd.concat(dfs, ignore_index=True)
# Commented this out so it runs smoothly for you
'''

'\nlistof_alldfs = ["season_2010_11_full.csv","season_2011_12_full.csv","season_2012_13_full.csv","season_2013_14_full.csv",\n                 "season_2014_15_full.csv","season_2015_16_full.csv","season_2016_17_full.csv","season_2017_18_full.csv",\n                 "season_2018_19_full.csv","season_2019_20_full.csv","season_2020_21_full.csv","season_2021_22_full.csv",\n"season_2022_23_full.csv","season_2023_24_full.csv","season_2024_25_full.csv"]\ndfs = [pd.read_csv(f) for f in listof_alldfs]\n\nfull_df = pd.concat(dfs, ignore_index=True)\n# Commented this out so it runs smoothly for you\n'

### cXIX.)  Identify Rookies and Select those

In [34]:
#print(full_df.head())

In [35]:
#print(full_df.columns.tolist())

In [36]:
# Commented this out so it runs smoothly for you
'''
# Compute first NBA season where the player played any games (defining FIRST_SEASON)
first_season_played = (
    full_df[full_df["GP"] > 0]
    .groupby("PLAYER_ID")["SEASON"]
    .min()
    .reset_index()
)

first_season_played.columns = ["PLAYER_ID", "FIRST_SEASON"]
# Commented this out so it runs smoothly for you
'''

'\n# Compute first NBA season where the player played any games (defining FIRST_SEASON)\nfirst_season_played = (\n    full_df[full_df["GP"] > 0]\n    .groupby("PLAYER_ID")["SEASON"]\n    .min()\n    .reset_index()\n)\n\nfirst_season_played.columns = ["PLAYER_ID", "FIRST_SEASON"]\n# Commented this out so it runs smoothly for you\n'

In [37]:
# Commented this out so it runs smoothly for you
'''
# Adding players' first season year into the data set
full_df = full_df.merge(first_season_played, on="PLAYER_ID", how="left")
# Commented this out so it runs smoothly for you
'''

'\n# Adding players\' first season year into the data set\nfull_df = full_df.merge(first_season_played, on="PLAYER_ID", how="left")\n# Commented this out so it runs smoothly for you\n'

In [38]:
# Commented this out so it runs smoothly for you
'''
# Filtering so only rookies are left in this dataset
rookies_df = full_df[
    (full_df["SEASON"] == full_df["FIRST_SEASON"]) &
    (full_df["GP"] > 0)
]
# Commented this out so it runs smoothly for you
'''

'\n# Filtering so only rookies are left in this dataset\nrookies_df = full_df[\n    (full_df["SEASON"] == full_df["FIRST_SEASON"]) &\n    (full_df["GP"] > 0)\n]\n# Commented this out so it runs smoothly for you\n'

In [39]:
#print(rookies_df)

## 1d) Specifying Who Received ROY Votes

#### dI.) Inputting Vote Shares

In [40]:
# Commented this out so it runs smoothly for you
'''
# Source: basketball-reference.com

# Initializing Everyone with a 0:
rookies_df["Vote_Share"] = 0

# 2024-25:
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Stephon Castle", "Vote_Share"] = 0.964
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Zaccharie Risacher", "Vote_Share"] = 0.490
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Jaylen Wells", "Vote_Share"] = 0.246
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Alex Sarr", "Vote_Share"] = 0.068
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Zach Edey", "Vote_Share"] = 0.020
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Kel\'el Ware", "Vote_Share"] = 0.008    ### check!
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Matas Buzelis", "Vote_Share"] = 0.002
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Jared McCain", "Vote_Share"] = 0.002

# 2023-24:
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Victor Wembanyama", "Vote_Share"] = 1.000
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Chet Holmgren", "Vote_Share"] = 0.596
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Brandon Miller", "Vote_Share"] = 0.174
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Jaime Jaquez Jr.", "Vote_Share"] = 0.020    ### check!
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Brandin Podziemski", "Vote_Share"] = 0.008
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Dereck Lively II", "Vote_Share"] = 0.002    ### check!

# 2022-23:
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Paolo Banchero", "Vote_Share"] = 0.988
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Jalen Williams", "Vote_Share"] = 0.482
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Walker Kessler", "Vote_Share"] = 0.228
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Bennedict Mathurin", "Vote_Share"] = 0.054
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Keegan Murray", "Vote_Share"] = 0.042
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Jaden Ivey", "Vote_Share"] = 0.006

# 2021-22:
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Scottie Barnes", "Vote_Share"] = 0.756
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Evan Mobley", "Vote_Share"] = 0.726
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Cade Cunningham", "Vote_Share"] = 0.306
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Jalen Green", "Vote_Share"] = 0.006
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Franz Wagner", "Vote_Share"] = 0.004
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Herbert Jones", "Vote_Share"] = 0.002

# 2020-21:
rookies_df.loc[rookies_df["PLAYER_NAME"]=="LaMelo Ball", "Vote_Share"] = 0.939      ### check!
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Anthony Edwards", "Vote_Share"] = 0.624
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Tyrese Haliburton", "Vote_Share"] = 0.230
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Saddiq Bey", "Vote_Share"] = 0.006

# 2019-20:
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Ja Morant", "Vote_Share"] = 0.996
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Kendrick Nunn", "Vote_Share"] = 0.408
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Zion Williamson", "Vote_Share"] = 0.280
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Brandon Clarke", "Vote_Share"] = 0.100
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Coby White", "Vote_Share"] = 0.006
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Terence Davis", "Vote_Share"] = 0.004
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Eric Paschall", "Vote_Share"] = 0.004
rookies_df.loc[rookies_df["PLAYER_NAME"]=="RJ Barett", "Vote_Share"] = 0.002

# 2018-19:
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Luka Dončić", "Vote_Share"] = 0.992
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Trae Young", "Vote_Share"] = 0.602
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Deandre Ayton", "Vote_Share"] = 0.132
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Jaren Jackson Jr.", "Vote_Share"] = 0.024
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Collin Sexton", "Vote_Share"] = 0.020
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Shai Gilgeous-Alexander", "Vote_Share"] = 0.014
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Marvin Bagley III", "Vote_Share"] = 0.012
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Josh Okogie", "Vote_Share"] = 0.002
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Mitchell Robinson", "Vote_Share"] = 0.002

# 2017-18:
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Ben Simmons", "Vote_Share"] = 0.952
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Donovan Mitchell", "Vote_Share"] = 0.640
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Jayson Tatum", "Vote_Share"] = 0.200
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Kyle Kuzma", "Vote_Share"] = 0.006
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Dennis Smith Jr.", "Vote_Share"] = 0.002

# 2016-17:
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Malcolm Brogdon", "Vote_Share"] = 0.828
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Dario Šarić", "Vote_Share"] = 0.532
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Joel Embiid", "Vote_Share"] = 0.354
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Buddy Hield", "Vote_Share"] = 0.042
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Willy Hernangómez", "Vote_Share"] = 0.016
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Jamal Murray", "Vote_Share"] = 0.016
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Marquese Chriss", "Vote_Share"] = 0.006
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Jaylen Brown", "Vote_Share"] = 0.002
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Yogi Ferrell", "Vote_Share"] = 0.002
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Rodney McGruder", "Vote_Share"] = 0.002

# 2015-16:
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Karl-Anthony Towns", "Vote_Share"] = 1.000
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Kristaps Porziņģis", "Vote_Share"] = 0.558 ### check!!!
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Nikola Jokić", "Vote_Share"] = 0.091
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Devin Booker", "Vote_Share"] = 0.075
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Jahlil Okafor", "Vote_Share"] = 0.052
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Justise Winslow", "Vote_Share"] = 0.011
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Emmanuel Mudiay", "Vote_Share"] = 0.006
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Myles Turner", "Vote_Share"] = 0.005
rookies_df.loc[rookies_df["PLAYER_NAME"]=="D'Angelo Russell", "Vote_Share"] = 0.002

# 2014-15:
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Andrew Wiggins", "Vote_Share"] = 0.929
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Nikola Mirotić", "Vote_Share"] = 0.515
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Nerlens Noel", "Vote_Share"] = 0.217
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Elfrid Payton", "Vote_Share"] = 0.122
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Marcus Smart", "Vote_Share"] = 0.009
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Jusuf Nurkić", "Vote_Share"] = 0.005
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Jordan Clarkson", "Vote_Share"] = 0.003

# 2013-14:
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Michael Carter-Williams", "Vote_Share"] = 0.918
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Victor Oladipo", "Vote_Share"] = 0.587
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Trey Burke", "Vote_Share"] = 0.155
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Mason Plumlee", "Vote_Share"] = 0.094
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Tim Hardaway Jr.", "Vote_Share"] = 0.037
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Gorgui Dieng", "Vote_Share"] = 0.005
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Steven Adams", "Vote_Share"] = 0.002
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Giannis Antetokounmpo", "Vote_Share"] = 0.002
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Nick Calathes", "Vote_Share"] = 0.002

# 2012-13:
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Damian Lillard", "Vote_Share"] = 1.000
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Anthony Davis", "Vote_Share"] = 0.506
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Bradley Beal", "Vote_Share"] = 0.155
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Andre Drummond", "Vote_Share"] = 0.060
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Dion Waiters", "Vote_Share"] = 0.035
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Harrison Barnes", "Vote_Share"] = 0.013
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Chris Copeland", "Vote_Share"] = 0.013
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Michael Kidd-Gilchrist", "Vote_Share"] = 0.005
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Jonas Valančiūnas", "Vote_Share"] = 0.003   #### check!!
rookies_df.loc[rookies_df["PLAYER_NAME"]=="John Jenkins", "Vote_Share"] = 0.002

# 2011-12:
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Kyrie Irving", "Vote_Share"] = 0.987
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Ricky Rubio", "Vote_Share"] = 0.283
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Kenneth Faried", "Vote_Share"] = 0.215
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Kawhi Leonard", "Vote_Share"] = 0.078
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Iman Shumpert", "Vote_Share"] = 0.055
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Klay Thompson", "Vote_Share"] = 0.050
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Isaiah Thomas", "Vote_Share"] = 0.047
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Brandon Knight", "Vote_Share"] = 0.035
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Chandler Parsons", "Vote_Share"] = 0.023
rookies_df.loc[rookies_df["PLAYER_NAME"]=="MarShon Brooks", "Vote_Share"] = 0.007
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Kemba Walker", "Vote_Share"] = 0.005
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Josh Selby", "Vote_Share"] = 0.002

# 2010-11:
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Blake Griffin", "Vote_Share"] = 1.000
rookies_df.loc[rookies_df["PLAYER_NAME"]=="John Wall", "Vote_Share"] = 0.500
rookies_df.loc[rookies_df["PLAYER_NAME"]=="DeMarcus Cousins", "Vote_Share"] = 0.137
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Landry Fields", "Vote_Share"] = 0.105
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Gary Neal", "Vote_Share"] = 0.032
rookies_df.loc[rookies_df["PLAYER_NAME"]=="Greg Monroe", "Vote_Share"] = 0.025

#print(rookies_df["Vote_Share"])
# Commented this out so it runs smoothly for you
'''

'\n# Source: basketball-reference.com\n\n# Initializing Everyone with a 0:\nrookies_df["Vote_Share"] = 0\n\n# 2024-25:\nrookies_df.loc[rookies_df["PLAYER_NAME"]=="Stephon Castle", "Vote_Share"] = 0.964\nrookies_df.loc[rookies_df["PLAYER_NAME"]=="Zaccharie Risacher", "Vote_Share"] = 0.490\nrookies_df.loc[rookies_df["PLAYER_NAME"]=="Jaylen Wells", "Vote_Share"] = 0.246\nrookies_df.loc[rookies_df["PLAYER_NAME"]=="Alex Sarr", "Vote_Share"] = 0.068\nrookies_df.loc[rookies_df["PLAYER_NAME"]=="Zach Edey", "Vote_Share"] = 0.020\nrookies_df.loc[rookies_df["PLAYER_NAME"]=="Kel\'el Ware", "Vote_Share"] = 0.008    ### check!\nrookies_df.loc[rookies_df["PLAYER_NAME"]=="Matas Buzelis", "Vote_Share"] = 0.002\nrookies_df.loc[rookies_df["PLAYER_NAME"]=="Jared McCain", "Vote_Share"] = 0.002\n\n# 2023-24:\nrookies_df.loc[rookies_df["PLAYER_NAME"]=="Victor Wembanyama", "Vote_Share"] = 1.000\nrookies_df.loc[rookies_df["PLAYER_NAME"]=="Chet Holmgren", "Vote_Share"] = 0.596\nrookies_df.loc[rookies_df["PLAYER

#### dII.) Correcting for undrafted players
##### For the purposes of modeling, I am alright with essentially saying that undrafted players were just drafted at the 61st overall position

In [41]:
# Commented this out so it runs smoothly for you
'''
rookies_df.loc[rookies_df["DRAFT_NUMBER"]=="Undrafted", "DRAFT_NUMBER"] = 61
rookies_df["DRAFT_NUMBER"] = rookies_df["DRAFT_NUMBER"].fillna(61)
rookies_df["DRAFT_NUMBER"] = pd.to_numeric(
    rookies_df["DRAFT_NUMBER"],
    errors="coerce"
)
rookies_df["DRAFT_BUCKET"] = rookies_df["DRAFT_NUMBER"].clip(upper=15)
# Commented this out so it runs smoothly for you
'''

'\nrookies_df.loc[rookies_df["DRAFT_NUMBER"]=="Undrafted", "DRAFT_NUMBER"] = 61\nrookies_df["DRAFT_NUMBER"] = rookies_df["DRAFT_NUMBER"].fillna(61)\nrookies_df["DRAFT_NUMBER"] = pd.to_numeric(\n    rookies_df["DRAFT_NUMBER"],\n    errors="coerce"\n)\nrookies_df["DRAFT_BUCKET"] = rookies_df["DRAFT_NUMBER"].clip(upper=15)\n# Commented this out so it runs smoothly for you\n'

In [42]:
#winns = np.where(rookies_df["Vote_Share"] != 0)
#
#for i in winns:
#    print(rookies_df.iloc[i][["PLAYER_NAME","Vote_Share"]])
#    print('\n')

#### dIII.) Saving this rookie dataset now

In [43]:
# Commented this out so it runs smoothly for you
'''
rookies_df.to_csv("historic_rookies.csv")
# Commented this out so it runs smoothly for you
'''

'\nrookies_df.to_csv("historic_rookies.csv")\n# Commented this out so it runs smoothly for you\n'

## 2. Feature Engineering

### 2a) Selecting the Final Features

In [44]:
FINAL_FEATURES = [
    "PTS", "REB", "AST", "STL",
    "BLK", "TOV", "MIN", "PLUS_MINUS", 
    'PRA',  "GP_pct", "Opp_met", "DRAFT_BUCKET"
#    ,'TS_pct', 'USG_pct', 'WS', 'TRB_pct', 'VORP',
]

### 2b) Handling Missing Data & Loading Datasets

In [45]:
# Loading the dfs from csvs
rookies_df = pd.read_csv('historic_rookies.csv')
rookies_2025_26 = pd.read_csv('rookies_2025_26.csv')

# Ensure we are working with a real dataframe, not a slice
rookies_df = rookies_df.copy()

# Impute missing values safely
rookies_df.loc[:, FINAL_FEATURES] = (
    rookies_df[FINAL_FEATURES]
    .fillna(rookies_df[FINAL_FEATURES].mean(numeric_only=True))
)

### 2c) Creating X & y sets

In [46]:
X = rookies_df[FINAL_FEATURES]
y = rookies_df["Vote_Share"]

## 3. Random Forest Model Training

### 3a) Data Split

In [47]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

### 3b) Completing Training

In [48]:
from sklearn.ensemble import RandomForestRegressor

time1 = time.time()

model = RandomForestRegressor(
    n_estimators=400,
    max_depth=8,
    min_samples_split=8,
    min_samples_leaf=3,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
time2 = time.time()
timfr = (time2-time1)/60
print('This took',round(timfr,3), 'minutes to run.')

This took 0.005 minutes to run.


## 4) Validating Performance
### 4a) Calculating metrics to understand performance: MAE, R^2, Spearman Rank

In [49]:
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("R²:", r2)

MAE: 0.019612652402526305
R²: 0.06904596509147043


In [50]:
from scipy.stats import spearmanr

spearman_corr, _ = spearmanr(y_test, y_pred)
print("Spearman Rank:", spearman_corr)

Spearman Rank: 0.3189972115060906


### 4b) Extract Feature Importance

In [51]:
pd.DataFrame({
    "Feature": FINAL_FEATURES,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

,Feature,Importance
5,TOV,0.142122
11,DRAFT_BUCKET,0.140987
0,PTS,0.139873
8,PRA,0.137167
2,AST,0.090306
6,MIN,0.073284
10,Opp_met,0.060899
7,PLUS_MINUS,0.058062
1,REB,0.045058
9,GP_pct,0.042480


## 5. Prediction Generation!

#### 5a) Prepping Current Rookie Matrix

In [59]:
X_current = rookies_2025_26[FINAL_FEATURES]

# Get feature column order from training
feature_cols = X_train.columns

# Reindex prediction dataframe to match training features
X_current = X_current.reindex(columns=feature_cols, fill_value=0)

#### 5b) Predicting Probabilities

In [60]:
# Predict expected vote share for current rookies
predicted_vote_share = model.predict(X_current)

# Remove any tiny negative values
predicted_vote_share = np.clip(predicted_vote_share, 0, None)

# Normalize into probabilities summing to 1
probabilities = predicted_vote_share / predicted_vote_share.sum()

# Build final table
predictions = pd.DataFrame({
    "player_name": rookies_2025_26["PLAYER_NAME"],
    "probability": probabilities
})

# Keep only positive probabilities
predictions = predictions[predictions["probability"] > 0]

# Rank by probability (descending)
predictions = predictions.sort_values(
    by="probability",
    ascending=False
).reset_index(drop=True)

# Set index to ranking starting at 1
predictions.index += 1

In [61]:
predictions['probability'] = predictions['probability'].round(6)

In [62]:
# If one wishes to print the output in the Jupyter Notebook
pd.set_option('display.max_rows', None)
print(predictions)
# Optionally, reset the option to its default after use
pd.reset_option('display.max_rows')

                 player_name  probability
1               Cooper Flagg     0.284689
2               Kon Knueppel     0.269926
3               VJ Edgecombe     0.164329
4             Jeremiah Fears     0.122439
5                Derik Queen     0.054147
6               Dylan Harper     0.040086
7              Cedric Coward     0.031625
8           Ryan Kalkbrenner     0.007782
9                 Sion James     0.005997
10                Ace Bailey     0.004606
11             Ryan Nembhard     0.003859
12               Tre Johnson     0.003757
13                Egor Dëmin     0.002982
14              Will Richard     0.001332
15          Hunter Dickinson     0.000672
16                 Ben Saraf     0.000254
17              Moussa Cisse     0.000189
18                Caleb Love     0.000181
19            Nique Clifford     0.000170
20            Dylan Cardwell     0.000163
21        Walter Clayton Jr.     0.000067
22      Collin Murray-Boyles     0.000054
23             Hugo González     0

In [63]:
# Verifying probability normalizes to 1
print(np.sum(predictions["probability"]))

0.9999960000000001


#### 5c) Exporting to CSV

In [64]:
#predictions.to_csv("predictions.csv", index=True)